## imports

In [16]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## importing the data

In [17]:
trans = pd.read_csv("../BankData/trans.csv",delimiter=";")

trans

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,930101,PRIJEM,VKLAD,700.0,700.0,NaN,NaN,NaN
1,171812,576,930101,PRIJEM,VKLAD,900.0,900.0,NaN,NaN,NaN
2,207264,704,930101,PRIJEM,VKLAD,1000.0,1000.0,NaN,NaN,NaN
3,1117247,3818,930101,PRIJEM,VKLAD,600.0,600.0,NaN,NaN,NaN
4,579373,1972,930102,PRIJEM,VKLAD,400.0,400.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1056315,3626622,2906,981231,PRIJEM,NaN,62.3,13729.4,UROK,NaN,NaN
1056316,3627616,2935,981231,PRIJEM,NaN,81.3,19544.9,UROK,NaN,NaN
1056317,3625403,2869,981231,PRIJEM,NaN,60.2,14638.2,UROK,NaN,NaN
1056318,3626683,2907,981231,PRIJEM,NaN,107.5,23453.0,UROK,NaN,NaN


In [18]:
trans.isna().sum()

trans_id           0
account_id         0
date               0
type               0
operation     183114
amount             0
balance            0
k_symbol      481881
bank          782812
account       760931
dtype: int64

In [19]:
trans.describe()

,trans_id,account_id,date,amount,balance,account
count,1.056320e+06,1.056320e+06,1.056320e+06,1.056320e+06,1.056320e+06,2.953890e+05
mean,1.335311e+06,2.936867e+03,9.656748e+05,5.924146e+03,3.851833e+04,4.567092e+07
std,1.227487e+06,2.477345e+03,1.394535e+04,9.522735e+03,2.211787e+04,3.066340e+07
min,1.000000e+00,1.000000e+00,9.301010e+05,0.000000e+00,-4.112570e+04,0.000000e+00
25%,4.302628e+05,1.204000e+03,9.601160e+05,1.359000e+02,2.240250e+04,1.782858e+07
50%,8.585065e+05,2.434000e+03,9.704100e+05,2.100000e+03,3.314340e+04,4.575095e+07
75%,2.060979e+06,3.660000e+03,9.802280e+05,6.800000e+03,4.960362e+04,7.201341e+07
max,3.682987e+06,1.138200e+04,9.812310e+05,8.740000e+04,2.096370e+05,9.999420e+07


## mapping the k_symbol

In [20]:
k_symbol_map = {
    'POJISTNE': 'insurance_payment',
    'SLUZBY': 'statement_payment',
    'UROK': 'interest_credited',
    'SANKC. UROK': 'sanction_interest',
    'SIPO': 'household_payment',
    'DUCHOD': 'pension_payment',
    'UVER': 'loan_payment',
    'LEASING': 'leasing'
}

# 2. Clean whitespace and map
# (Replace `trans` with your DataFrame name if you are working on another table)
trans['k_symbol'] = trans['k_symbol'].astype(str).str.strip()

trans['k_symbol'] = (
    trans['k_symbol']
    .map(k_symbol_map)
    .fillna('other_unknown')
    .replace({'nan': 'other_unknown', '': 'other_unknown'})
)

# 3. Verify value counts
trans['k_symbol'].value_counts()

k_symbol
other_unknown        535314
interest_credited    183114
statement_payment    155832
household_payment    118065
pension_payment       30338
insurance_payment     18500
loan_payment          13580
sanction_interest      1577
Name: count, dtype: int64

## mapping operations

In [21]:
operation_map = {
    'VYBER KARTOU': 'credit_card_withdrawal',
    'VKLAD': 'credit_in_cash',
    'PREVOD Z UCTU': 'collection_other_bank',
    'VYBER': 'withdrawal_in_cash',
    'PREVOD NA UCET': 'remittance_to_other_bank'
}

# 1. Clean whitespace and map
trans['operation'] = trans['operation'].astype(str).str.strip()

trans['operation'] = (
    trans['operation']
    .map(operation_map)
    .fillna('other_unknown')
    .replace({'nan': 'other_unknown', '': 'other_unknown'})
)

# 2. Check counts
trans['operation'].value_counts()

operation
withdrawal_in_cash          434918
remittance_to_other_bank    208283
other_unknown               183114
credit_in_cash              156743
collection_other_bank        65226
credit_card_withdrawal        8036
Name: count, dtype: int64

## mapping type

In [22]:
type_map = {
    'PRIJEM': 'credit',
    'VYBER': 'withdrawal',
    'VYDAJ': 'withdrawal'
}

# Clean whitespace and apply mapping
trans['type'] = trans['type'].astype(str).str.strip()

trans['type'] = (
    trans['type']
    .map(type_map)
    .fillna('other_unknown')
    .replace({'nan': 'other_unknown', '': 'other_unknown'})
)

# Verify distribution
trans['type'].value_counts()

type
withdrawal    651237
credit        405083
Name: count, dtype: int64

## parsing the date

In [23]:
trans['date'] = pd.to_datetime(trans['date'].astype(str).str.zfill(6), format='%y%m%d')

In [49]:
trans.sample(10)

,trans_id,account_id,date,type,operation,amount,balance,k_symbol
938019,447363,1519,1998-08-10,withdrawal,remittance_to_other_bank,3443.0,73920.8,household_payment
467423,765357,2614,1997-01-17,withdrawal,withdrawal_in_cash,390.0,14787.5,other_unknown
479021,3449912,4034,1997-01-31,credit,other_unknown,175.0,42599.7,interest_credited
253242,405838,1381,1996-01-01,withdrawal,withdrawal_in_cash,2320.0,16201.6,other_unknown
86633,1143223,3909,1994-09-06,withdrawal,withdrawal_in_cash,12800.0,41298.5,other_unknown
778229,328369,1120,1998-02-05,credit,collection_other_bank,5428.0,26685.4,pension_payment
505632,744716,2542,1997-03-08,withdrawal,remittance_to_other_bank,308.0,25690.8,insurance_payment
131474,224495,764,1995-01-28,withdrawal,withdrawal_in_cash,3900.0,23830.2,other_unknown
854630,1097381,3750,1998-05-06,credit,credit_in_cash,20233.0,73239.1,other_unknown
815774,977426,3330,1998-03-24,withdrawal,withdrawal_in_cash,22000.0,42350.0,other_unknown


In [25]:
trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056320 entries, 0 to 1056319
Data columns (total 10 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   trans_id    1056320 non-null  int64         
 1   account_id  1056320 non-null  int64         
 2   date        1056320 non-null  datetime64[ns]
 3   type        1056320 non-null  object        
 4   operation   1056320 non-null  object        
 5   amount      1056320 non-null  float64       
 6   balance     1056320 non-null  float64       
 7   k_symbol    1056320 non-null  object        
 8   bank        273508 non-null   object        
 9   account     295389 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 80.6+ MB


In [26]:
trans["bank"].value_counts()

bank
QR    22285
AB    21720
ST    21711
YZ    21582
GH    21499
EF    21293
KL    21234
UV    21167
OP    21094
IJ    20525
WX    20178
MN    19623
CD    19597
Name: count, dtype: int64

In [32]:
trans = trans.drop(columns=['bank', 'account'])

In [33]:
trans.head()

,trans_id,account_id,date,type,operation,amount,balance,k_symbol
0,695247,2378,1993-01-01,credit,credit_in_cash,700.0,700.0,other_unknown
1,171812,576,1993-01-01,credit,credit_in_cash,900.0,900.0,other_unknown
2,207264,704,1993-01-01,credit,credit_in_cash,1000.0,1000.0,other_unknown
3,1117247,3818,1993-01-01,credit,credit_in_cash,600.0,600.0,other_unknown
4,579373,1972,1993-01-02,credit,credit_in_cash,400.0,400.0,other_unknown


In [41]:
trans["type"].value_counts()

type
withdrawal    651237
credit        405083
Name: count, dtype: int64

In [42]:
trans["operation"].value_counts()

operation
withdrawal_in_cash          434918
remittance_to_other_bank    208283
other_unknown               183114
credit_in_cash              156743
collection_other_bank        65226
credit_card_withdrawal        8036
Name: count, dtype: int64

In [43]:
trans["k_symbol"].value_counts()

k_symbol
other_unknown        535314
interest_credited    183114
statement_payment    155832
household_payment    118065
pension_payment       30338
insurance_payment     18500
loan_payment          13580
sanction_interest      1577
Name: count, dtype: int64

In [46]:
trans[trans["type"] == "withdrawal" ]

,trans_id,account_id,date,type,operation,amount,balance,k_symbol
136,695560,2378,1993-01-31,withdrawal,withdrawal_in_cash,34700.0,59453.7,other_unknown
189,145127,485,1993-02-03,withdrawal,withdrawal_in_cash,25300.0,46177.9,other_unknown
222,1028004,3510,1993-02-06,withdrawal,withdrawal_in_cash,1600.0,22271.0,other_unknown
233,695340,2378,1993-02-07,withdrawal,remittance_to_other_bank,9612.0,80033.7,household_payment
249,622121,2122,1993-02-07,withdrawal,withdrawal_in_cash,14100.0,32859.1,other_unknown
...,...,...,...,...,...,...,...,...
1055233,3607597,2346,1998-12-31,withdrawal,withdrawal_in_cash,14.9,-2236.2,sanction_interest
1055579,3668638,5066,1998-12-31,withdrawal,withdrawal_in_cash,3.5,59667.7,sanction_interest
1055639,3489430,7520,1998-12-31,withdrawal,withdrawal_in_cash,67.2,-11020.4,sanction_interest
1055676,3488804,7465,1998-12-31,withdrawal,withdrawal_in_cash,0.8,84975.2,sanction_interest


In [39]:
trans[trans["balance"] < 0 ]

,trans_id,account_id,date,type,operation,amount,balance,k_symbol
4703,1176458,4034,1993-06-28,withdrawal,withdrawal_in_cash,31840.0,-2994.6,other_unknown
4719,1693860,5740,1993-06-28,withdrawal,withdrawal_in_cash,10500.0,-3724.7,other_unknown
4808,1176550,4034,1993-06-30,withdrawal,withdrawal_in_cash,14.6,-2906.6,statement_payment
5026,3449868,4034,1993-06-30,withdrawal,withdrawal_in_cash,3.0,-2909.6,sanction_interest
5027,3449867,4034,1993-06-30,credit,other_unknown,102.5,-2892.0,interest_credited
...,...,...,...,...,...,...,...,...
1055216,3607199,2335,1998-12-31,withdrawal,withdrawal_in_cash,36.3,-2838.9,sanction_interest
1055220,3607198,2335,1998-12-31,credit,other_unknown,6.6,-2785.5,interest_credited
1055233,3607597,2346,1998-12-31,withdrawal,withdrawal_in_cash,14.9,-2236.2,sanction_interest
1055235,3607596,2346,1998-12-31,credit,other_unknown,16.6,-2221.3,interest_credited


In [50]:
trans.to_csv("../cleanData/clean_trans.csv", sep=";", index=False)